# Synthetic Data Evaluation

## What This Is
Synthetic data evaluation measures three properties:

1. **Fidelity**: Does the synthetic data look like real data statistically?
2. **Utility**: Can models trained on synthetic data perform well on real tasks?
3. **Privacy**: Does the synthetic data protect the privacy of real individuals?

## Key Metrics

### Fidelity
- **Column-wise statistics**: Mean, std, percentiles per column
- **Wasserstein distance**: Distribution distance for continuous columns
- **TVD (Total Variation Distance)**: Distribution distance for categorical columns
- **Correlation difference**: How well inter-column correlations are preserved

### Utility
- **Train-on-Synthetic Test-on-Real (TSTR)**: Train ML model on synth, test on real
- **Train-on-Real Test-on-Real (TRTR)**: Baseline performance with real training data
- **TSTR/TRTR ratio**: Utility preservation factor

### Privacy
- **Distance to Closest Record (DCR)**: Min distance from each synthetic record to any real record
- **Nearest Neighbor Distance Ratio (NNDR)**: DCR normalized by second-nearest neighbor distance
- **Membership inference on synthetic**: Can MIA detect real records from synthetic ones?

In [ ]:
import numpy as np
from typing import Dict, List, Tuple

np.random.seed(42)

# Generate real and synthetic datasets
N = 500
age_r = np.random.normal(40, 12, N).clip(18, 80)
income_r = (20000 + age_r * 1000 + np.random.normal(0, 8000, N)).clip(15000, 200000)
credit_r = (500 + income_r / 1000 + np.random.normal(0, 50, N)).clip(300, 850)
loan_r = (income_r * 0.3 * np.random.uniform(0.5, 2.0, N)).clip(1000, 100000)
default_r = ((loan_r / income_r > 0.4) & (credit_r < 650)).astype(int)

real = {'age': age_r, 'income': income_r, 'credit_score': credit_r,
        'loan_amount': loan_r, 'default': default_r}

# Synthetic: good synthesis (preserves correlations)
age_s = age_r[np.random.choice(N, N)] + np.random.randn(N) * 3
income_s = (20000 + age_s * 1000 + np.random.normal(0, 9000, N)).clip(15000, 200000)
credit_s = (500 + income_s / 1000 + np.random.normal(0, 55, N)).clip(300, 850)
loan_s = (income_s * 0.3 * np.random.uniform(0.5, 2.0, N)).clip(1000, 100000)
default_s = ((loan_s / income_s > 0.4) & (credit_s < 650)).astype(int)

synth = {'age': age_s, 'income': income_s, 'credit_score': credit_s,
         'loan_amount': loan_s, 'default': default_s}

print('Real and synthetic datasets ready.')
print(f'Correlation (real income-credit):  {np.corrcoef(income_r, credit_r)[0,1]:.3f}')
print(f'Correlation (synth income-credit): {np.corrcoef(income_s, credit_s)[0,1]:.3f}')


In [ ]:
# Fidelity metrics

def wasserstein_1d(a, b, n_bins=100):
    sa, sb = np.sort(a), np.sort(b)
    n = min(len(sa), len(sb))
    return np.mean(np.abs(sa[:n] - sb[:n]))

def correlation_matrix_diff(real: Dict, synth: Dict, cols: List[str]) -> float:
    """Frobenius norm of correlation matrix difference."""
    R_real = np.corrcoef(np.column_stack([real[c] for c in cols]))
    R_synth = np.corrcoef(np.column_stack([synth[c] for c in cols]))
    return np.linalg.norm(R_real - R_synth, 'fro')

cont_cols = ['age', 'income', 'credit_score', 'loan_amount']

print('Fidelity Report')
print('=' * 60)
print(f'{'Column':<15} {'W1 Distance':>12} {'Mean Diff %':>12}')
print('-' * 42)

for col in cont_cols:
    w1 = wasserstein_1d(real[col], synth[col])
    mean_diff_pct = abs(real[col].mean() - synth[col].mean()) / real[col].mean() * 100
    print(f'{col:<15} {w1:>12.2f} {mean_diff_pct:>11.2f}%')

corr_diff = correlation_matrix_diff(real, synth, cont_cols)
print(f'\nCorrelation matrix difference (Frobenius): {corr_diff:.4f}')
print(f'Default rate - real: {real["default"].mean():.3f}, synth: {synth["default"].mean():.3f}')


In [ ]:
# Utility: Train-on-Synthetic Test-on-Real (TSTR)

def sigmoid(z): return 1 / (1 + np.exp(-np.clip(z, -500, 500)))

def train_lr(X, y, lr=0.05, epochs=200):
    w = np.zeros(X.shape[1])
    b = 0.0
    for _ in range(epochs):
        err = sigmoid(X @ w + b) - y
        w -= lr * (X.T @ err) / len(y)
        b -= lr * err.mean()
    return w, b

def accuracy(X, y, w, b): return ((sigmoid(X @ w + b) > 0.5).astype(int) == y).mean()

feature_cols = ['age', 'income', 'credit_score', 'loan_amount']

def to_X(data): return np.column_stack([data[c] for c in feature_cols])

# Normalize features
X_real = to_X(real)
X_synth = to_X(synth)
mu, sigma = X_real.mean(0), X_real.std(0) + 1
X_real_n = (X_real - mu) / sigma
X_synth_n = (X_synth - mu) / sigma

# Split real into train/test
X_r_train, y_r_train = X_real_n[:400], real['default'][:400]
X_r_test, y_r_test = X_real_n[400:], real['default'][400:]

# TRTR: Train on Real, Test on Real
w_trtr, b_trtr = train_lr(X_r_train, y_r_train)
acc_trtr = accuracy(X_r_test, y_r_test, w_trtr, b_trtr)

# TSTR: Train on Synthetic, Test on Real
w_tstr, b_tstr = train_lr(X_synth_n, synth['default'])
acc_tstr = accuracy(X_r_test, y_r_test, w_tstr, b_tstr)

print('Utility Report (TSTR)')
print('=' * 40)
print(f'TRTR (real train → real test): {acc_trtr:.3f}')
print(f'TSTR (synth train → real test): {acc_tstr:.3f}')
print(f'Utility ratio: {acc_tstr/acc_trtr:.3f}')
print(f'(1.0 = perfect utility; <0.9 = significant utility loss)')


In [ ]:
# Privacy: Distance to Closest Record (DCR) and NNDR

def compute_dcr(synth_X: np.ndarray, real_X: np.ndarray, sample_size: int = 200) -> np.ndarray:
    """For each synthetic record, find minimum distance to any real record."""
    # Subsample for speed
    s_idx = np.random.choice(len(synth_X), min(sample_size, len(synth_X)), replace=False)
    r_idx = np.random.choice(len(real_X), min(sample_size, len(real_X)), replace=False)
    
    S = synth_X[s_idx]
    R = real_X[r_idx]
    
    dcrs = []
    for s in S:
        dists = np.linalg.norm(R - s, axis=1)
        dcrs.append(dists.min())
    return np.array(dcrs)

def compute_nndr(synth_X: np.ndarray, real_X: np.ndarray, sample_size: int = 100) -> np.ndarray:
    """NNDR: DCR / distance to second-nearest neighbor. Low = privacy risk."""
    s_idx = np.random.choice(len(synth_X), min(sample_size, len(synth_X)), replace=False)
    r_idx = np.random.choice(len(real_X), min(sample_size, len(real_X)), replace=False)
    S = synth_X[s_idx]
    R = real_X[r_idx]
    
    nndrs = []
    for s in S:
        dists = np.sort(np.linalg.norm(R - s, axis=1))
        if len(dists) >= 2 and dists[1] > 1e-8:
            nndrs.append(dists[0] / dists[1])
    return np.array(nndrs)

dcr = compute_dcr(X_synth_n, X_real_n)
nndr = compute_nndr(X_synth_n, X_real_n)

print('Privacy Report')
print('=' * 40)
print(f'DCR  - mean: {dcr.mean():.4f}, 5th pct: {np.percentile(dcr, 5):.4f}')
print(f'NNDR - mean: {nndr.mean():.4f}, 5th pct: {np.percentile(nndr, 5):.4f}')
print()
print('Interpretation:')
print('  DCR 5th pct near 0 → some synthetic records very close to real records → privacy risk')
print('  NNDR near 1 → nearest neighbors equidistant → potential memorization')
print('  NNDR 5th pct > 0.5 generally considered acceptable')
